# Recovering Neighborhood Boundaries from Public Data — Pittsburgh Demo

This notebook is the runnable companion to [`docs/WRITEUP.md`](../docs/WRITEUP.md). It runs the full pipeline end-to-end on a small sub-region of Pittsburgh, PA, using **only public data**:

- OpenStreetMap (street network, barriers, amenities) via `osmnx`
- US Census ACS 5-year estimates via the Census API (free key required — see below)
- City of Pittsburgh's published 90-neighborhood boundary file, used as ground truth for validation

**Requires:** a free Census API key from https://api.census.gov/data/key_signup.html, set as the `CENSUS_API_KEY` environment variable before running.

This is a generalized technique demo, not a production pipeline. See the main [README's Scope section](../README.md#scope--whats-not-here) for what's deliberately left out.

## 1. Define the study area

We use a sub-region of Pittsburgh rather than the whole city, to keep the demo fast and the data pull small. The chosen area spans a few well-known, distinctly-named neighborhoods and crosses at least one real physical barrier (a river or a freeway corridor), since that's the interesting case for the barrier-aware connectivity step later.

In [ ]:
import os
import sys
sys.path.insert(0, "../src")

import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt

from pipeline import (
    generate_hex_grid, join_signals_to_hexes,
    build_adjacency_graph, apply_barrier_cuts, add_barrier_side_feature,
    run_constrained_ward,
    compute_purity, compute_coherence,
    repair_articulation_points, smooth_boundaries_topological,
)

# Sub-region: adjust this place name/bbox to taste. Using a named OSM place
# query keeps this reproducible without hardcoding a bounding box.
STUDY_AREA_NAME = "Pittsburgh, Pennsylvania, USA"  # narrow this to a sub-area as desired

study_area = ox.geocode_to_gdf(STUDY_AREA_NAME)
study_area.plot(figsize=(6, 6), edgecolor="black", facecolor="none")
plt.title("Study area boundary")
plt.show()

## 2. Pull ground truth: Pittsburgh's published neighborhood boundaries

Pittsburgh is a useful demo city specifically because it has an official, city-maintained neighborhood boundary file — most US cities don't. This lets us actually score the clustering output against something real, rather than eyeballing it.

Source: City of Pittsburgh's open data portal (Western PA Regional Data Center / WPRDC), neighborhood boundary layer.

In [ ]:
# City of Pittsburgh neighborhood boundaries — public WPRDC dataset.
# (URL/resource ID may need updating if WPRDC reorganizes their catalog;
# check https://data.wprdc.org for the current "Neighborhoods" dataset.)
PGH_NEIGHBORHOODS_URL = "https://data.wprdc.org/dataset/neighborhoods2/resource/<RESOURCE_ID>"

# ground_truth = gpd.read_file(PGH_NEIGHBORHOODS_URL)
# ground_truth = ground_truth.rename(columns={"hood": "neighborhood_id"})
# ground_truth = ground_truth.clip(study_area)
# ground_truth.plot(figsize=(6, 6), column="neighborhood_id", legend=False)

# Placeholder until the live resource ID is filled in:
ground_truth = None
print("TODO: fill in WPRDC resource ID, then uncomment the load above")

## 3. Build the H3 hex grid and join signal data

We tile the study area into H3 hexagons, then join two kinds of public signal:

- **ACS demographic/economic signal** (income, density, housing age) via the Census API, joined onto hexes by area-weighted overlap with block groups
- **OSM amenity counts** (restaurants, parks, transit stops) via `osmnx`, counted per hex

Resolution 9 hexes are roughly 0.1 km² — fine enough for neighborhood-scale clustering without excessive sparse-data coverage gaps.

In [ ]:
hexes = generate_hex_grid(study_area.geometry.iloc[0], resolution=9)
print(f"{len(hexes)} hexes generated")
hexes.plot(figsize=(6, 6), edgecolor="white", linewidth=0.2)
plt.title("H3 hex grid over study area")
plt.show()

In [ ]:
# --- ACS signal pull (requires CENSUS_API_KEY env var) ---
from census import Census
from us import states

# c = Census(os.environ["CENSUS_API_KEY"])
# acs_data = c.acs5.state_county_blockgroup(
#     fields=("B19013_001E", "B25035_001E", "B01003_001E"),  # median income, median year built, population
#     state_fips=states.PA.fips,
#     county_fips="003",  # Allegheny County
#     blockgroup="*",
# )
# Join acs_data (as block group polygons from TIGER/Line) onto `hexes` via join_signals_to_hexes(...)

print("TODO: wire up Census API pull, join via pipeline.join_signals_to_hexes")

In [ ]:
# --- OSM amenity counts ---
# tags = {"amenity": ["restaurant", "cafe", "park"], "shop": True}
# amenities = ox.features_from_polygon(study_area.geometry.iloc[0], tags)
# amenity_counts_per_hex = ...  # spatial join + groupby count, by hex_id

print("TODO: pull OSM amenities, aggregate counts per hex")

## 4. Identify physical barriers

Pull freeway and river geometry from OSM, to be used both for cutting adjacency edges and for the signed barrier-side feature described in the writeup.

In [ ]:
# barriers = ox.features_from_polygon(
#     study_area.geometry.iloc[0],
#     tags={"highway": ["motorway", "trunk"], "waterway": "river"},
# )
# barriers = barriers[barriers.geometry.type.isin(["LineString", "MultiLineString"])]

print("TODO: pull barrier geometry from OSM")

## 5. Build the constrained adjacency graph

Standard H3 grid adjacency, with edges crossing a barrier removed, plus a signed barrier-side distance feature appended to the signal matrix.

In [ ]:
# adjacency = build_adjacency_graph(hexes)
# adjacency = apply_barrier_cuts(adjacency, hexes, barriers)
# barrier_side = add_barrier_side_feature(hexes, barriers)
# hexes["barrier_side"] = barrier_side

print("TODO: build adjacency, apply barrier cuts, add barrier-side feature")

## 6. Cluster

Run Ward agglomerative clustering, constrained to the barrier-aware adjacency graph. `N_CLUSTERS` is the key tradeoff parameter described in the writeup — try a few values and watch how purity and coherence move in opposite directions.

In [ ]:
SIGNAL_COLUMNS = []  # fill in with the ACS + amenity + barrier_side columns from steps 3-5
N_CLUSTERS = 8  # tune for this small sub-region; see writeup for the purity/coherence tradeoff

# features = hexes[SIGNAL_COLUMNS].fillna(hexes[SIGNAL_COLUMNS].median()).values
# labels = run_constrained_ward(features, adjacency, n_clusters=N_CLUSTERS)
# hexes["district_id"] = labels

print("TODO: run clustering once steps 3-5 are wired up")

## 7. Validate against ground truth

Compute purity (did a real neighborhood get fragmented?) and coherence (did a district swallow multiple real neighborhoods?) against Pittsburgh's published boundary file.

In [ ]:
# district_polys = hexes.dissolve(by="district_id").reset_index()
# purity_df = compute_purity(district_polys, ground_truth)
# coherence_df = compute_coherence(district_polys, ground_truth)

# print("Median purity:", purity_df["purity"].median())
# print("Median coherence:", coherence_df["coherence"].median())
# purity_df.sort_values("purity").head(10)

print("TODO: run validation once clustering output exists")

## 8. Repair and smooth

Fix dumbbell-shaped districts via articulation-point splitting, then simplify boundaries via the topological shared-edge approach so adjacent districts stay seam-free.

In [ ]:
# hex_adjacency_dict = {...}  # build from H3 grid_disk per hex_id, see connectivity.build_adjacency_graph
# repaired_labels = repair_articulation_points(hexes, hexes["district_id"].values, hex_adjacency_dict)
# hexes["district_id"] = repaired_labels

# district_polys = hexes.dissolve(by="district_id").reset_index()
# district_polys_projected = district_polys.to_crs(district_polys.estimate_utm_crs())
# smoothed = smooth_boundaries_topological(district_polys_projected, tolerance=15)  # meters
# smoothed = smoothed.to_crs("EPSG:4326")

print("TODO: run repair + smoothing once clustering output exists")

## 9. Plot the final result

Side-by-side: algorithmic output vs. published ground truth, for visual sanity-checking against the purity/coherence numbers above.

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(14, 7))
# smoothed.plot(ax=axes[0], column="district_id", legend=False, edgecolor="white", linewidth=0.5)
# axes[0].set_title(f"Algorithmic districts (k={N_CLUSTERS})")
# ground_truth.plot(ax=axes[1], column="neighborhood_id", legend=False, edgecolor="white", linewidth=0.5)
# axes[1].set_title("Published Pittsburgh neighborhoods (ground truth)")
# plt.tight_layout()
# plt.show()

print("TODO: final comparison plot")

## Next steps

This notebook stops at the algorithmic output. As discussed in the writeup, the lowest-purity/lowest-coherence districts from step 7 are the ones worth a structured manual review pass — not something to keep tuning `N_CLUSTERS` against indefinitely. See `docs/WRITEUP.md` for more on that tradeoff and where this technique sits in a larger product pipeline.